In [5]:
# Install ultralytics if not already installed
!pip install ultralytics

import cv2
from ultralytics import settings, YOLO
from google.colab.patches import cv2_imshow
from google.colab.output import eval_js
from IPython.display import display, Javascript
import numpy as np
import base64  # Corrected import

# Disable data and crash report tracking
settings.update({"sync": False})

# Load the lightweight YOLOv8 Medium model (more accurate than nano)
model = YOLO('yolov8m.pt')

# JavaScript function to inject and run the webcam stream
def video_stream():
  js = Javascript('''
    var video;
    var container = document.createElement('div');
    var captureCanvas = document.createElement('canvas');
    var img = document.createElement('img');

    async function initWebcam() {
      video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(container);
      container.appendChild(video);
      container.appendChild(img);

      video.srcObject = stream;
      await video.play();

      // Resize the output canvas to match video stream
      captureCanvas.width = video.videoWidth;
      captureCanvas.height = video.videoHeight;
    }

    async function streamFrame() {
      var ctx = captureCanvas.getContext('2d');
      ctx.drawImage(video, 0, 0);
      return captureCanvas.toDataURL('image/jpeg', 0.8);
    }

    function showProcessed(imgData) {
      img.src = imgData;
      video.style.display = 'none'; // Hide raw video, show annotated image
    }
    ''')
  display(js)

# Helper to convert JS base64 image back into an OpenCV image matrix
def js_to_image(js_reply):
  image_bytes = eval_js('streamFrame()')
  encoded_data = image_bytes.split(',')
  nparr = np.frombuffer(base64.b64decode(encoded_data[1]), np.uint8)
  img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
  return img

# Helper to convert processed OpenCV frame back to base64 for JS display
def image_to_js(cv_img):
  _, encoded_img = cv2.imencode('.jpg', cv_img)
  base64_img = base64.b64encode(encoded_img).decode('utf-8')
  return f"data:image/jpeg;base64,{base64_img}"

# Start the webcam stream in JavaScript
video_stream()
eval_js('initWebcam()')

print("Starting Object Counter... Stop execution in Colab to quit.")

while True:
    try:
        # 1. Capture frame from the browser webcam
        frame = js_to_image(None)

        # 2. Perform object detection
        results = model(frame, verbose=False)
        detected_count = len(results[0].boxes)

        # 3. Render detection boxes
        annotated_frame = results[0].plot()

        # 4. Display the object count overlay
        cv2.putText(
            annotated_frame,
            f'Total Objects Counted: {detected_count}',
            (20, 50),
            cv2.FONT_HERSHEY_SIMPLEX, 1,
            (0, 255, 0), 2, cv2.LINE_AA
        )

        # 5. Send the annotated frame back to the browser interface
        js_img_data = image_to_js(annotated_frame)
        eval_js(f'showProcessed("{js_img_data}")')

    except Exception as e:
        print(f"Stream ended or interrupted: {e}")
        break

<IPython.core.display.Javascript object>

Starting Object Counter... Stop execution in Colab to quit.


KeyboardInterrupt: 